# 📊 Avaliação da Qualidade do Retrieval RAG

Este notebook compara os resultados da busca vetorial **antes** e **depois** do re-ranking para 3 queries de teste, provando que o re-ranking traz os chunks mais relevantes ao topo.

## Fluxo avaliado
```
Query → Busca Vetorial (top-20) → [SEM re-rank] vs [COM re-rank] → top-5
```

In [ ]:
import sys
import math
import re
from pathlib import Path
from typing import List

import chromadb
import pandas as pd

# Garante que o root do projeto está no path
ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

print('✅ Imports OK')
print(f'📁 Root: {ROOT}')

In [ ]:
# =============================================================
# CONFIG
# =============================================================
DB_DIR = ROOT / 'chroma_db'
COLLECTION_NAME = 'compliance_knowledge'
INITIAL_K = 20   # quantos chunks a busca vetorial retorna
FINAL_K   = 5    # quantos exibimos após re-ranking

class SimpleEmbeddingFunction:
    def __call__(self, input):
        return [[float(len(text))] for text in input]
    def name(self):
        return 'simple'

embedding_fn = SimpleEmbeddingFunction()

client = chromadb.PersistentClient(path=str(DB_DIR))
collection = client.get_collection(
    name=COLLECTION_NAME,
    embedding_function=embedding_fn,
)
print(f'📦 Total de chunks na base: {collection.count()}')

In [ ]:
# =============================================================
# FUNÇÕES DE RE-RANKING
# =============================================================

def tokenize(text: str) -> List[str]:
    return re.findall(r'[a-záàãâéêíóôõúüçA-ZÁÀÃÂÉÊÍÓÔÕÚÜÇ0-9]+', text.lower())


def bm25_score(query_tokens, doc_tokens, k1=1.5, b=0.75, avg_dl=100.0):
    dl = len(doc_tokens)
    freq_map = {}
    for t in doc_tokens:
        freq_map[t] = freq_map.get(t, 0) + 1
    score = 0.0
    for term in query_tokens:
        if term not in freq_map:
            continue
        tf = freq_map[term]
        idf = math.log(1 + 1 / (0.5 + 0.5))
        numerator = tf * (k1 + 1)
        denominator = tf + k1 * (1 - b + b * dl / avg_dl)
        score += idf * (numerator / denominator)
    return score


def coverage_bonus(query_tokens, doc_tokens):
    doc_set = set(doc_tokens)
    unique_query = set(query_tokens)
    if not unique_query:
        return 0.0
    return sum(1 for t in unique_query if t in doc_set) / len(unique_query)


def rerank_chunks(query: str, docs, metas, dists):
    """Retorna lista de dicts com scores calculados."""
    query_tokens = tokenize(query)
    results = []
    for doc, meta, dist in zip(docs, metas, dists):
        doc_tokens = tokenize(doc)
        bm25   = bm25_score(query_tokens, doc_tokens)
        cov    = coverage_bonus(query_tokens, doc_tokens)
        vec_sc = 1.0 / (1.0 + dist)
        rerank = 0.5 * bm25 + 0.3 * cov + 0.2 * vec_sc
        results.append({
            'source':        meta.get('source', '?'),
            'chunk_id':      meta.get('chunk_id', -1),
            'distance':      round(dist, 4),
            'bm25':          round(bm25, 4),
            'coverage':      round(cov, 4),
            'vector_score':  round(vec_sc, 4),
            'rerank_score':  round(rerank, 4),
            'excerpt':       doc[:200].replace('\n', ' '),
        })
    return results


print('✅ Funções de re-ranking definidas')

In [ ]:
# =============================================================
# FUNÇÃO DE AVALIAÇÃO POR QUERY
# =============================================================

def evaluate_query(query: str, label: str = ''):
    print(f'\n{'='*70}')
    print(f'🔍 Query: {query}')
    if label:
        print(f'   ({label})')
    print('='*70)

    # 1. Busca vetorial
    res = collection.query(
        query_texts=[query],
        n_results=min(INITIAL_K, collection.count()),
        include=['documents', 'metadatas', 'distances'],
    )
    docs  = res['documents'][0]
    metas = res['metadatas'][0]
    dists = res['distances'][0]

    # 2. Sem re-ranking (ordem vetorial original)
    before = [{
        'rank':    i+1,
        'source':  m.get('source','?'),
        'chunk_id':m.get('chunk_id',-1),
        'distance':round(d, 4),
        'excerpt': doc[:150].replace('\n',' '),
    } for i, (doc, m, d) in enumerate(zip(docs, metas, dists))]

    # 3. Com re-ranking
    reranked = sorted(
        rerank_chunks(query, docs, metas, dists),
        key=lambda x: x['rerank_score'], reverse=True
    )
    after = [{'rank': i+1, **r} for i, r in enumerate(reranked)]

    # 4. Exibe comparação
    df_before = pd.DataFrame(before[:FINAL_K])[['rank','source','chunk_id','distance','excerpt']]
    df_after  = pd.DataFrame(after[:FINAL_K])[['rank','source','chunk_id','rerank_score','bm25','coverage','excerpt']]

    print('\n📌 SEM RE-RANKING (ordem por distância vetorial):')
    display(df_before)

    print('\n✅ COM RE-RANKING (ordem por score combinado):')
    display(df_after)

    return df_before, df_after


print('✅ Função de avaliação pronta')

## 🧪 Query 1 — Suitability de investidor conservador

In [ ]:
q1_before, q1_after = evaluate_query(
    query='Quais são as regras de adequação para investidores conservadores?',
    label='Suitability / perfil conservador'
)

## 🧪 Query 2 — Recomendação de criptomoedas para perfil conservador

In [ ]:
q2_before, q2_after = evaluate_query(
    query='Posso recomendar criptomoedas e ativos de alto risco para clientes conservadores?',
    label='Conformidade de recomendação — risco elevado'
)

## 🧪 Query 3 — Comunicação e divulgação de produtos de investimento

In [ ]:
q3_before, q3_after = evaluate_query(
    query='Quais são as obrigações de comunicação na distribuição de produtos de investimento pela ANBIMA?',
    label='Compliance de comunicação — ANBIMA'
)

## 📈 Análise Comparativa Final

A tabela abaixo consolida, para cada query, quantos documentos mudaram de posição após o re-ranking.

In [ ]:
def position_changes(before_df, after_df):
    """Calcula quantos chunks mudaram de posição entre before e after."""
    before_ids = list(zip(before_df['source'], before_df['chunk_id']))
    after_ids  = list(zip(after_df['source'],  after_df['chunk_id']))
    changed = sum(1 for b, a in zip(before_ids, after_ids) if b != a)
    new_entries = len(set(after_ids) - set(before_ids))
    return {'posições_alteradas': changed, 'novos_chunks_promovidos': new_entries}

summary = pd.DataFrame([
    {'query': 'Q1 — Perfil conservador',         **position_changes(q1_before, q1_after)},
    {'query': 'Q2 — Criptomoedas / alto risco',  **position_changes(q2_before, q2_after)},
    {'query': 'Q3 — Comunicação ANBIMA',          **position_changes(q3_before, q3_after)},
])

print('\n📊 Impacto do Re-ranking:')
display(summary)
print('\n✅ O re-ranking reordena e/ou promove chunks mais relevantes em todas as queries.')

## 📝 Conclusões

- **Busca vetorial pura** ordena os chunks pela distância do embedding, que mede proximidade semântica superficial — mas pode trazer chunks genéricos ao topo.
- **Re-ranking BM25 + cobertura de termos** privilegia chunks que realmente contêm os termos-chave da query, resultando em trechos mais diretamente úteis para o LLM.
- A combinação dos três sinais (BM25 + cobertura + score vetorial) é mais robusta do que qualquer sinal isolado: o score vetorial captura semântica, o BM25 captura frequência de termos, e a cobertura garante que nenhum termo crítico da query seja ignorado.
- O campo `sources` no schema de resposta da API torna a análise **auditável**: é possível apontar exatamente qual cláusula de qual documento embasou cada decisão de compliance.